# Ouro-1.4B → HumanEval (build · train · eval) — Colab / Ampere

Keystone OS Σ₀ coder. This notebook runs the **whole** flywheel on a cloud GPU because the
local RTX 3070 (8 GB) can *train* the recurrent Ouro LoopLM but **cannot decode/eval** it
(a 16-token probe didn't finish in 450 s — it swaps). On an **A100 or L4** the stock cached
`generate()` is fast, so you get a real **pass@1**.

**Before you run:** `Runtime → Change runtime type → A100` (or **L4**). High-RAM helps.
- A100 / L4 (cc ≥ 8.0): bf16 + 4-bit QLoRA — good for **train and eval**.
- T4 (cc 7.5): no bf16 → fine for **eval** (fp16), but **don't train** here (fp16 QLoRA on this
  reasoning LM risks a NaN adapter).

Flow: **① deps → ② config → ③ build corpus → ④ train → ⑤ eval → ⑥ (optional) push to HF**.
Set `TRAIN=False` + point `ADAPTER` at an existing HF repo/path to **eval-only**.

**Free-Colab T4?** Don't train here (no bf16 → NaN-adapter risk). Set `PRESET="t4-eval"` in ② to
load an existing adapter and get a real pass@1 — see the config cell.

## ① GPU check + dependencies

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NO GPU — set Runtime → GPU")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.57,<4.58", "peft>=0.10", "bitsandbytes>=0.43",
    "datasets", "accelerate", "scipy", "huggingface_hub"], check=True)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| cc", torch.cuda.get_device_capability() if torch.cuda.is_available() else None)

## ② Config

`ADAPTER_OUT` is where training writes. To **eval an existing adapter** instead of training,
set `TRAIN=False` and `ADAPTER='<hf-repo-or-path>'` (e.g. `lanternfounder/ouro-checkpoints`).

In [ ]:
# --- quick preset (pick your GPU) ---
# "full"    = build corpus + train + eval — needs an Ampere GPU (A100/L4); bf16 required.
# "t4-eval" = EVAL ONLY, safe on a free-Colab T4. T4 has no bf16, and fp16 QLoRA on this
#             recurrent reasoning LM risks a NaN adapter, so we DON'T train here. We load an
#             existing adapter and produce a real HumanEval pass@1 — the one thing the local
#             3070 can't do. Set ADAPTER="" in the preset for a base-model baseline instead.
PRESET = "full"          # "full" | "t4-eval"

# --- toggles ---
BUILD_CORPUS = True      # build the decontaminated HumanEval corpus
TRAIN        = True      # train the QLoRA adapter (set False to eval an existing one)
EVAL         = True      # run HumanEval pass@1
PUSH_TO_HUB  = False     # push the trained adapter to HF (needs HF_TOKEN, see last cell)

# --- params ---
BASE        = "ByteDance/Ouro-1.4B"
DATA        = "humaneval-train.jsonl"
ADAPTER_OUT = "ouro-humaneval-adapter"
ADAPTER     = ADAPTER_OUT          # what EVAL loads; override with an HF repo id / path for eval-only
CORPUS_LIMIT = 15000               # rows after dedup/decontam
MAX_STEPS   = 600                  # -1 = use EPOCHS; ~600 ≈ a solid run on A100
EPOCHS      = 1
SEQ         = 1280                 # corpus p99≈966, max≈1262 → no truncation
LR          = 2e-4
LORA_R      = 16
EVAL_N      = None                 # None = full 164; or an int for a quick subset
MAX_NEW     = 384
HF_PUSH_REPO = "lanternfounder/ouro-checkpoints"

# T4-safe eval-only: skip corpus build / train / push, eval an existing adapter.
if PRESET == "t4-eval":
    BUILD_CORPUS = TRAIN = PUSH_TO_HUB = False
    EVAL = True
    ADAPTER = "lanternfounder/ouro-checkpoints"   # HF repo/path to eval; "" = base-model baseline
    print("PRESET t4-eval -> eval-only, ADAPTER =", ADAPTER or "(base model)")

print("config set:", dict(BUILD_CORPUS=BUILD_CORPUS, TRAIN=TRAIN, EVAL=EVAL, PUSH_TO_HUB=PUSH_TO_HUB))

## ③ Build the decontaminated HumanEval corpus

Magicoder OSS-Instruct + bigcode self-oss-instruct (exec-filtered) + MBPP (train/val/prompt —
**test split excluded**, it's the benchmark). Decontaminated against HumanEval by 13-gram overlap,
char-length-capped so no code answer truncates mid-function. Mirrors `scripts/build_humaneval_corpus.py`.

In [ ]:
import json, re, random
from datasets import load_dataset

NGRAM, MAX_CHARS = 13, 5500
def _w(t): return re.sub(r"[^a-z0-9]+"," ",(t or "").lower()).split()
def _grams(ws,n=NGRAM):
    for i in range(len(ws)-n+1): yield " ".join(ws[i:i+n])

if BUILD_CORPUS:
    he = load_dataset("openai_humaneval", split="test")
    poison = set()
    for r in he:
        for f in ("prompt","canonical_solution"): poison.update(_grams(_w(r.get(f,""))))
    print(f"decontam: {len(poison)} HumanEval 13-grams")

    pool = []
    for r in load_dataset("google-research-datasets/mbpp","full",split="train"):
        t=(r.get("text") or "").strip(); c=(r.get("code") or "").strip(); tl=r.get("test_list") or []
        if t and c: pool.append({"instruction": t+("\n\nYour code should pass these tests:\n"+"\n".join(tl) if tl else ""),"input":"","output":c})
    for split in ("validation","prompt"):
        for r in load_dataset("google-research-datasets/mbpp","full",split=split):
            t=(r.get("text") or "").strip(); c=(r.get("code") or "").strip()
            if t and c: pool.append({"instruction":t,"input":"","output":c})
    print(f"mbpp: {len(pool)}")
    grab = int(CORPUS_LIMIT*0.75)+2000
    oss=[r for r in load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K",split="train") if (r.get("lang") or "").lower()=="python"][:grab]
    pool += [{"instruction":(r.get("problem") or "").strip(),"input":"","output":(r.get("solution") or "").strip()} for r in oss if r.get("problem") and r.get("solution")]
    soss=list(load_dataset("bigcode/self-oss-instruct-sc2-exec-filter-50k",split="train"))[:grab]
    pool += [{"instruction":(r.get("instruction") or "").strip(),"input":"","output":(r.get("response") or "").strip()} for r in soss if r.get("instruction") and r.get("response")]
    print(f"pool: {len(pool)}")

    random.seed(20260626); random.shuffle(pool)
    seen, kept, dup, lng, con = set(), [], 0, 0, 0
    for r in pool:
        k=r["instruction"][:200]
        if k in seen: dup+=1; continue
        seen.add(k)
        if len(r["instruction"])+len(r["output"])>MAX_CHARS: lng+=1; continue
        ws=_w(r["instruction"]+"\n"+r["output"])
        if len(ws)>=NGRAM and any(g in poison for g in _grams(ws)): con+=1; continue
        kept.append(r)
        if len(kept)>=CORPUS_LIMIT: break
    with open(DATA,"w",encoding="utf-8") as f:
        for r in kept: f.write(json.dumps(r,ensure_ascii=False)+"\n")
    print(f"wrote {len(kept)} rows -> {DATA} (dropped {dup} dup, {lng} long, {con} HumanEval-leaked)")

## ④ Train the QLoRA adapter

Mirrors `scripts/train-qlora-ouro.py`: 4-bit nf4 + bf16 compute (Ampere), LoRA on all-linear,
completion-only loss (prompt masked), and the ROPE patch Ouro's custom code needs on transformers ≥ 4.53.

In [ ]:
if TRAIN:
    import os, json, torch
    from datasets import Dataset
    from transformers import (AutoModelForCausalLM, AutoTokenizer, AutoConfig, BitsAndBytesConfig,
                              Trainer, TrainingArguments, default_data_collator)
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    try:
        from transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS, _compute_default_rope_parameters
        ROPE_INIT_FUNCTIONS.setdefault("default", _compute_default_rope_parameters)
    except Exception as e: print("rope patch skipped:", e)

    cc = torch.cuda.get_device_capability()
    use_bf16 = torch.cuda.is_bf16_supported() and cc >= (8,0)
    if not use_bf16:
        print("WARNING: bf16 unavailable (need Ampere). fp16 QLoRA on this reasoning LM risks a NaN adapter — use A100/L4 to train.")
    cdt = torch.bfloat16 if use_bf16 else torch.float16
    print(f"precision: {'bf16' if use_bf16 else 'fp16'} (cc {cc[0]}.{cc[1]})")

    tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(BASE, trust_remote_code=True)
    if getattr(cfg,"pad_token_id",None) is None: cfg.pad_token_id = getattr(cfg,"bos_token_id",tok.pad_token_id)
    model = AutoModelForCausalLM.from_pretrained(BASE, config=cfg, device_map="auto", trust_remote_code=True,
        attn_implementation="sdpa",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=cdt, bnb_4bit_use_double_quant=True))
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=2*LORA_R, lora_dropout=0.05,
        bias="none", task_type="CAUSAL_LM", target_modules="all-linear"))
    model.print_trainable_parameters()

    rows=[]
    for line in open(DATA,encoding="utf-8"):
        line=line.strip()
        if not line: continue
        r=json.loads(line)
        if r.get("instruction") and r.get("output"):
            rows.append({"text": f"### Instruction:\n{r['instruction']}\n\n### Response:\n{r['output']}{tok.eos_token}"})
    print("training rows:", len(rows))
    ds = Dataset.from_list(rows)
    def tok_fn(b):
        enc = tok(b["text"], truncation=True, max_length=SEQ, padding="max_length")
        labels=[]
        for text, ids, am in zip(b["text"], enc["input_ids"], enc["attention_mask"]):
            prompt = text.split("### Response:\n",1)[0] + "### Response:\n"
            plen = len(tok(prompt, truncation=True, max_length=SEQ)["input_ids"])
            labels.append([-100 if (am[j]==0 or j<plen) else ids[j] for j in range(len(ids))])
        enc["labels"]=labels; return enc
    ds = ds.map(tok_fn, batched=True, remove_columns=["text"])
    Trainer(model=model, train_dataset=ds, data_collator=default_data_collator,
        args=TrainingArguments(output_dir=ADAPTER_OUT, num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
            per_device_train_batch_size=1, gradient_accumulation_steps=8, learning_rate=LR,
            bf16=use_bf16, fp16=not use_bf16, max_grad_norm=1.0, warmup_ratio=0.03,
            logging_steps=10, save_strategy="steps", save_steps=200, save_total_limit=2,
            optim="paged_adamw_8bit", gradient_checkpointing=True, report_to="none")).train()
    final=os.path.join(ADAPTER_OUT,"final"); model.save_pretrained(final); tok.save_pretrained(final)
    ADAPTER=final; print("adapter saved ->", final)
    del model; torch.cuda.empty_cache()

## ⑤ Eval — HumanEval pass@1

Canonical 164-problem set with faithful sandboxed execution. Uses Ouro's **stock cached**
`generate()` (the fast product path). Mirrors `scripts/eval_humaneval_ouro.py`.

In [ ]:
if EVAL:
    import torch, re, json, subprocess, sys, tempfile, time, os
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from datasets import load_dataset
    STOP = ["\ndef ", "\nclass ", "\nif __name__", "\n\n\n", "\n```"]
    def trim_body(b):
        b=b.replace("\t","    ")
        for s in STOP:
            k=b.find(s)
            if k!=-1: b=b[:k]
        return b.rstrip()
    def make_candidate(text, ep, hp):
        m=re.search(r"```(?:python)?\s*(.*?)```", text, re.S)
        if m:
            c=m.group(1); i=c.find(f"def {ep}")
            if i>=0:
                di=hp.find(f"def {ep}"); pre=hp[:di] if di>=0 else ""
                lines=c[i:].splitlines(); blk=[lines[0]]
                for ln in lines[1:]:
                    if ln.strip()=="" or ln[:1] in (" ","\t"): blk.append(ln)
                    else: break
                cand=(pre+"\n".join(blk).rstrip()).replace("\t","    ")
                try: compile(cand,"<c>","exec"); return cand
                except SyntaxError: pass
            text=c
        body=trim_body(text)
        if not body.strip(): return None
        cand=hp+body
        try: compile(cand,"<c>","exec"); return cand
        except SyntaxError:
            lines=cand.splitlines()
            while lines:
                lines.pop()
                try: compile("\n".join(lines),"<c>","exec"); return "\n".join(lines)
                except SyntaxError: continue
            return None
    def run_test(cand, test, ep, timeout=12):
        if cand is None: return False,"no-parse"
        prog=cand+"\n\n"+test+f"\n\ncheck({ep})\n"
        with tempfile.NamedTemporaryFile("w",suffix=".py",delete=False,encoding="utf-8") as f:
            f.write(prog); p=f.name
        try:
            r=subprocess.run([sys.executable,p],capture_output=True,timeout=timeout,text=True)
            ok=r.returncode==0
            return ok, ("" if ok else (r.stderr.strip().splitlines() or ["?"])[-1][:120])
        except subprocess.TimeoutExpired: return False,"timeout"
        finally:
            try: os.unlink(p)
            except OSError: pass

    tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
    tok.pad_token = tok.bos_token   # Ouro: bos==eos; use bos as pad so generate() doesn't stop at token 0
    model = AutoModelForCausalLM.from_pretrained(BASE, trust_remote_code=True, dtype=torch.float16, device_map="auto")
    if ADAPTER:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, ADAPTER); print("loaded adapter:", ADAPTER)
    model.eval()

    ds = load_dataset("openai_humaneval", split="test")
    n = len(ds) if EVAL_N is None else min(EVAL_N, len(ds))
    n_ok, t0 = 0, time.time()
    print(f"\n{'task':<14}{'pass':<5}note")
    for i in range(n):
        ex=ds[i]
        ids=tok(ex["prompt"], return_tensors="pt").input_ids.to(model.device)
        with torch.no_grad():
            out=model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=MAX_NEW,
                do_sample=False, use_cache=True, repetition_penalty=1.1, pad_token_id=tok.pad_token_id,
                eos_token_id=None, stop_strings=STOP, tokenizer=tok)
        text=tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
        ok,note=run_test(make_candidate(text, ex["entry_point"], ex["prompt"]), ex["test"], ex["entry_point"])
        n_ok+=int(ok)
        print(f"{ex['task_id']:<14}{'OK ' if ok else 'x  ':<5}{note}")
    dt=time.time()-t0
    print(f"\nVERDICT HumanEval{'' if EVAL_N is None else f'[first {n}]'} "
          f"pass@1 = {100*n_ok/n:.1f}%  ({n_ok}/{n})  {dt/n:.1f}s/problem")

## ⑥ (Optional) Push the adapter to HF Hub

Set `PUSH_TO_HUB=True` above. Needs an HF token — add it as a Colab secret named `HF_TOKEN`
(🔑 in the left sidebar) or paste when prompted.

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import HfApi, login
    tokv=None
    try:
        from google.colab import userdata; tokv=userdata.get("HF_TOKEN")
    except Exception: pass
    if not tokv:
        import getpass; tokv=getpass.getpass("HF token: ")
    login(token=tokv)
    HfApi().upload_folder(folder_path=os.path.join(ADAPTER_OUT,"final"),
        repo_id=HF_PUSH_REPO, path_in_repo="humaneval-adapter", repo_type="model")
    print("pushed ->", HF_PUSH_REPO, "/humaneval-adapter")